<a href="https://colab.research.google.com/github/aldo02032004/naufaldo.github.io/blob/main/Phase_0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SEL 1: Install dependency + Import + Setup

In [22]:
!pip install -q pandas openpyxl requests

In [23]:
from google.colab import userdata

token = userdata.get('GH_TOKEN')
!pip install -q "great[all] @ git+https://{token}@github.com/azmkto/GREAT-Tools.git"

import os
import re
import html
import json
import time
import random
from io import BytesIO

import requests
import pandas as pd
from tqdm.auto import tqdm
from IPython.display import display

from great.text import normalize_slang

from google import genai
from google.genai import types

tqdm.pandas()

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [24]:
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
print("API key sudah di-set.")

gemini_client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])

# Daftar model fallback, urut dari yang paling diinginkan.
MODEL_CANDIDATES = [
    "gemini-flash-latest",
    "gemini-2.5-flash",
    "gemini-2.5-flash-lite",
]

GEMINI_MODEL = MODEL_CANDIDATES[0]  # default, berubah otomatis kalau fallback aktif


def get_gemini_response(text, max_retries=3, base_delay=5, models=None):
    """
    Kirim teks ke Gemini API. Untuk tiap model di 'models' (default:
    MODEL_CANDIDATES), coba sampai 'max_retries' kali dengan exponential
    backoff kalau kena 503/429. Kalau satu model gagal total, otomatis
    lanjut coba model berikutnya di daftar.
    """
    global GEMINI_MODEL

    if not isinstance(text, str) or text.strip() == "":
        return ""

    models_to_try = models or MODEL_CANDIDATES

    for model_name in models_to_try:
        for attempt in range(1, max_retries + 1):
            try:
                response = gemini_client.models.generate_content(
                    model=model_name,
                    contents=text,
                )
                if hasattr(response, "text") and response.text:
                    GEMINI_MODEL = model_name  # ingat model yang terakhir berhasil
                    return response.text.strip()
                else:
                    print(f"⚠️ Respons kosong dari model '{model_name}'.")
                    break  # coba model berikutnya

            except Exception as e:
                error_str = str(e)
                is_retryable = ("503" in error_str) or ("UNAVAILABLE" in error_str) or ("429" in error_str)
                is_not_found = ("404" in error_str) or ("NOT_FOUND" in error_str)

                if is_not_found:
                    print(f"⚠️ Model '{model_name}' tidak tersedia (404). Lanjut ke model berikutnya...")
                    break  # langsung skip ke model lain, tidak perlu retry

                if is_retryable and attempt < max_retries:
                    delay = base_delay * (2 ** (attempt - 1)) + random.uniform(0, 2)
                    print(f"⚠️ Model '{model_name}' percobaan {attempt}/{max_retries} gagal "
                          f"({error_str[:80]}...). Coba lagi dalam {delay:.1f} detik...")
                    time.sleep(delay)
                    continue
                else:
                    print(f"⚠️ Model '{model_name}' gagal setelah {attempt} percobaan "
                          f"({error_str[:80]}...). Lanjut ke model berikutnya...")
                    break  # habis retry untuk model ini, coba model lain

    print("⚠️ Semua model di daftar fallback gagal. Return string kosong.")
    return ""


# Uji coba singkat
test_response = get_gemini_response("Halo, tolong balas dengan satu kata saja.")
print(f"Model yang berhasil dipakai: {GEMINI_MODEL}")
print("Test response:", test_response)

API key sudah di-set.
⚠️ Model 'gemini-flash-latest' percobaan 1/3 gagal (429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your cu...). Coba lagi dalam 5.7 detik...
⚠️ Model 'gemini-flash-latest' percobaan 2/3 gagal (429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your cu...). Coba lagi dalam 10.3 detik...
⚠️ Model 'gemini-flash-latest' gagal setelah 3 percobaan (429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your cu...). Lanjut ke model berikutnya...
⚠️ Model 'gemini-2.5-flash' tidak tersedia (404). Lanjut ke model berikutnya...
⚠️ Model 'gemini-2.5-flash-lite' tidak tersedia (404). Lanjut ke model berikutnya...
⚠️ Semua model di daftar fallback gagal. Return string kosong.
Model yang berhasil dipakai: gemini-flash-latest
Test response: 


# SEL 2: Load data mentions dari Excel Drive

In [25]:
def resolve_xlsx_url(source: str) -> str:
    """Ubah link Google Sheets/Drive jadi url download .xlsx."""
    sheet_match = re.search(r"docs\.google\.com/spreadsheets/d/([a-zA-Z0-9-_]+)", source)
    drive_match = re.search(r"drive\.google\.com/file/d/([a-zA-Z0-9-_]+)", source)
    if sheet_match:
        return f"https://docs.google.com/spreadsheets/d/{sheet_match.group(1)}/export?format=xlsx"
    if drive_match:
        return f"https://drive.google.com/uc?export=download&id={drive_match.group(1)}"
    return source  # sudah berupa path/url .xlsx


def _download_bytes(url: str) -> BytesIO:
    session = requests.Session()
    headers = {"User-Agent": "Mozilla/5.0"}
    resp = session.get(url, headers=headers, allow_redirects=True)
    resp.raise_for_status()
    content_type = resp.headers.get("Content-Type", "")
    if "html" in content_type.lower():
        raise ValueError(
            "Gagal download file asli (dapat HTML, bukan xlsx). "
            "Pastikan akses file/sheet sudah 'Anyone with the link'."
        )
    return BytesIO(resp.content)


def _find_best_sheet(xls: pd.ExcelFile, expected_cols: list, skiprows: int):
    """Cari sheet yang headernya paling cocok dengan expected_cols."""
    best_sheet, best_match = xls.sheet_names[0], -1
    for name in xls.sheet_names:
        try:
            preview = pd.read_excel(xls, sheet_name=name, skiprows=skiprows, nrows=0)
            preview.columns = [c.strip() for c in preview.columns]
            match_count = sum(c in preview.columns for c in expected_cols)
            if match_count > best_match:
                best_match, best_sheet = match_count, name
        except Exception:
            continue
    return best_sheet


def load_raw_data(source: str, sheet_name=None, skiprows: int = 1) -> pd.DataFrame:
    url = resolve_xlsx_url(source)
    file_obj = _download_bytes(url) if url.startswith("http") else url
    xls = pd.ExcelFile(file_obj)

    expected_cols = ["No", "Headline", "Mentions", "Date", "Link", "Media", "Sentiment", "Author"]

    if sheet_name is None:
        sheet_name = _find_best_sheet(xls, expected_cols, skiprows)
        print(f"[INFO] Menggunakan sheet: '{sheet_name}'")

    df = pd.read_excel(xls, sheet_name=sheet_name, skiprows=skiprows)
    df.columns = [c.strip() for c in df.columns]

    missing = [c for c in expected_cols if c not in df.columns]
    if missing:
        print(f"[WARNING] Kolom hilang: {missing}")

    df = df.dropna(subset=["Mentions"]).reset_index(drop=True) if "Mentions" in df.columns else df
    available_cols = [c for c in expected_cols if c in df.columns]
    return df[available_cols]


SOURCE = "https://docs.google.com/spreadsheets/d/1Q4-Uknh0Dl_9KpqkvZpCIZYwlolsU63a/edit?usp=drive_link&ouid=116825097454650626545&rtpof=true&sd=true"

df_raw = load_raw_data(SOURCE)
print(f"Data berhasil dimuat: {df_raw.shape}")
display(df_raw.head())

[INFO] Menggunakan sheet: 'Sheet1'
Data berhasil dimuat: (8747, 8)


,No,Headline,Mentions,Date,Link,Media,Sentiment,Author
0,1,NaN,"RT PRABOWO DIHUJAT SE ASEAN, laporan yang masu...",2026-09-10 23:59:55,https://twitter.com/web/statuses/2098094128915...,Twitter,Negative,@lifecapture_
1,2,NaN,"RT PRABOWO DIHUJAT SE ASEAN, laporan yang masu...",2026-09-10 23:59:51,https://twitter.com/web/statuses/2098094113166...,Twitter,Negative,@jstcallmeaul
2,3,NaN,"RT PRABOWO DIHUJAT SE ASEAN, laporan yang masu...",2026-09-10 23:58:52,https://twitter.com/web/statuses/2098093865895...,Twitter,Negative,@takayatamiskin
3,4,Tiorita: Pemkab Langkat Siap Kolaborasi Dukung...,"FOTO BERSAMA : Pelaksana Tugas Bupati Langkat,...",2026-09-10 23:57:56,https://bbsnews.id/2026/09/10/tiorita-pemkab-l...,News,Positive,bbsnews.id
4,5,NaN,"RT Belum, belum over. War is over kalo prabowo...",2026-09-10 23:56:36,https://twitter.com/web/statuses/2098093293938...,Twitter,Negative,@jstcallmeaul


# SEL 3: Data Cleaning

In [26]:
URL_PATTERN = re.compile(r'https?://\S+|www\.\S+')
HASHTAG_PATTERN = re.compile(r'#\S+')
MENTION_PATTERN = re.compile(r'@\S+')

# Pola emoji (mencakup mayoritas rentang unicode emoji)
EMOJI_PATTERN = re.compile(
    "["
    "\U0001F300-\U0001FAFF"  # symbols & pictographs, extended
    "\U0001F1E6-\U0001F1FF"  # flags
    "\U00002700-\U000027BF"  # dingbats
    "\U0001F900-\U0001F9FF"  # supplemental symbols
    "\U00002600-\U000026FF"  # misc symbols
    "\U0001FA70-\U0001FAFF"
    "\U00002190-\U000021FF"  # arrows (sering dipakai spam)
    "]+", flags=re.UNICODE
)

BOILERPLATE_PATTERNS = [
    r"Dilarang keras mengambil konten.*?ANTARA\.",
    r"Kirim\s*Komentar menjadi tanggung-?jawab Anda sesuai UU ITE\.",
    r"Yuk Subscribe.*?(?=\n\n|\Z)",
    r"Follow (WA Channel|our Official).*?(?=\n\n|\Z)",
    r"Like our Official Facebook.*?(?=\n\n|\Z)",
    r"Uploader\s*:\s*\S+",
    r"Tanggal Tayang\s*:.*",
    r"Selengkapnya baca di\s*:\s*\S+",
    r"rt",
]
BOILERPLATE_RE = re.compile("|".join(BOILERPLATE_PATTERNS), flags=re.IGNORECASE | re.DOTALL)


def clean_text(raw_text: str) -> str:
    if not isinstance(raw_text, str):
        return ""
    text = html.unescape(raw_text)
    text = BOILERPLATE_RE.sub(" ", text)
    text = URL_PATTERN.sub(" ", text)
    text = HASHTAG_PATTERN.sub(" ", text)
    text = MENTION_PATTERN.sub(" ", text)
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{2,}', '\n', text)
    return text.strip()


def is_low_quality_text(text: str, min_words: int = 3, min_chars: int = 10) -> bool:
    """
    Mendeteksi teks yang isinya cuma emoji/simbol/tanda baca tanpa substansi.
    Return True jika teks dianggap tidak penting (harus dibuang).
    """
    if not isinstance(text, str) or text.strip() == "":
        return True

    no_emoji = EMOJI_PATTERN.sub(" ", text)
    # Buang tanda baca & karakter non-alfanumerik, sisakan huruf/angka/spasi
    stripped = re.sub(r'[^\w\s]', ' ', no_emoji, flags=re.UNICODE)
    stripped = re.sub(r'\s+', ' ', stripped).strip()

    word_count = len(stripped.split())
    char_count = len(stripped)

    return (word_count < min_words) or (char_count < min_chars)


# ---------------------------------------------------------
# Proses pembersihan dasar
# ---------------------------------------------------------
df_clean = df_raw.copy()

print("[INFO] Membersihkan teks...")
df_clean["clean_text"] = df_clean["Mentions"].progress_apply(clean_text)
df_clean = df_clean[df_clean["clean_text"].str.len() > 20].reset_index(drop=True)

# ---------------------------------------------------------
# FILTER SPAM #1: Teks emoji-only / tidak substantif
# ---------------------------------------------------------
print("[INFO] Mendeteksi teks emoji-only / tidak penting...")
df_clean["is_low_quality"] = df_clean["clean_text"].progress_apply(is_low_quality_text)

before_lq = len(df_clean)
df_clean = df_clean[~df_clean["is_low_quality"]].reset_index(drop=True)
print(f"[INFO] Filter emoji/tidak-penting: {before_lq} -> {len(df_clean)} baris "
      f"({before_lq - len(df_clean)} baris dibuang)")
df_clean = df_clean.drop(columns="is_low_quality")

df_clean["doc_id"] = [f"DOC-{i:05d}" for i in range(len(df_clean))]

# ---------------------------------------------------------
# FILTER SPAM #2: Post berulang-ulang (repetitive/bot spam)
# ---------------------------------------------------------
# Ambang batas: jika teks (dinormalisasi) muncul lebih dari MAX_REPEAT kali,
# dianggap broadcast/spam (bukan mentions organik) dan seluruh kemunculannya dibuang.
MAX_REPEAT = 5

df_clean["repeat_key"] = df_clean["clean_text"].astype(str).str.strip().str.lower()
repeat_counts = df_clean["repeat_key"].value_counts()
spam_keys = repeat_counts[repeat_counts > MAX_REPEAT].index

before_spam = len(df_clean)
df_clean["is_spam_repetitive"] = df_clean["repeat_key"].isin(spam_keys)
n_spam_rows = df_clean["is_spam_repetitive"].sum()
n_spam_unique = len(spam_keys)

df_clean = df_clean[~df_clean["is_spam_repetitive"]].reset_index(drop=True)
print(f"[INFO] Filter spam repetitif (>{MAX_REPEAT}x): {before_spam} -> {len(df_clean)} baris "
      f"({n_spam_rows} baris dari {n_spam_unique} teks unik dibuang)")

df_clean = df_clean.drop(columns=["repeat_key", "is_spam_repetitive"])

# ---------------------------------------------------------
# Dedupe (headline + awal clean_text)
# ---------------------------------------------------------
df_clean["dedupe_key"] = (
    df_clean["Headline"].astype(str).str.strip().str.lower() + "||" +
    df_clean["clean_text"].astype(str).str[:200].str.strip().str.lower()
)
before = len(df_clean)
df_clean = df_clean.drop_duplicates(subset="dedupe_key", keep="first").reset_index(drop=True)
print(f"[INFO] Dedup: {before} -> {len(df_clean)} baris")
df_clean = df_clean.drop(columns="dedupe_key")

# ---------------------------------------------------------
# Normalisasi teks (lowercase, slang)
# ---------------------------------------------------------
df_clean["search_text"] = df_clean["Headline"].str.lower().fillna("") + " " + df_clean["clean_text"].fillna("")
df_clean["clean_text"] = df_clean["clean_text"].str.lower()
df_clean["search_text"] = df_clean["search_text"].str.lower()

print("[INFO] Normalisasi slang...")
df_clean["search_text"] = df_clean["search_text"].progress_apply(normalize_slang)

print(f"\n[INFO] Pembersihan & filter spam selesai. Ukuran data akhir: {df_clean.shape}")
display(df_clean[["doc_id", "Mentions", "clean_text", "search_text", "Link", "Sentiment", "Author"]].head(5))

[INFO] Membersihkan teks...


  0%|          | 0/8747 [00:00<?, ?it/s]

[INFO] Mendeteksi teks emoji-only / tidak penting...


  0%|          | 0/8338 [00:00<?, ?it/s]

[INFO] Filter emoji/tidak-penting: 8338 -> 8332 baris (6 baris dibuang)
[INFO] Filter spam repetitif (>5x): 8332 -> 5253 baris (3079 baris dari 201 teks unik dibuang)
[INFO] Dedup: 5253 -> 3866 baris
[INFO] Normalisasi slang...


  0%|          | 0/3866 [00:00<?, ?it/s]


[INFO] Pembersihan & filter spam selesai. Ukuran data akhir: (3866, 11)


,doc_id,Mentions,clean_text,search_text,Link,Sentiment,Author
0,DOC-00003,"FOTO BERSAMA : Pelaksana Tugas Bupati Langkat,...","foto bersama : pelaksana tugas bupati langkat,...",tiorita: pemkab langkat siap kolaborasi dukung...,https://bbsnews.id/2026/09/10/tiorita-pemkab-l...,Positive,bbsnews.id
1,DOC-00007,Sebanyak 11 kapal gabungan telah diterjunkan u...,sebanyak 11 kapal gabungan telah diterjunkan u...,prabowo perintahkan tni al optimalkan pencaria...,https://news.republika.co.id/berita/tl5qxq393/...,Neutral,news.republika.co.id
2,DOC-00011,PRABOWO MANTAP&quot;PENYERAHAN BANTUAN BECA LI...,"prabowo mantap""penyerahan bantuan beca listrik...",prabowo bagus&quot;penyerahan bantuan beca lis...,https://www.youtube.com/watch?v=BXMdYvVoOuA,Positive,SAHABAT KANG DEDI MULYADI69
3,DOC-00012,@RA_leather ketika dirugikan atas perbuatan me...,ketika dirugikan atas perbuatan melawan hukum ...,ketika dirugikan atas perbuatan melawan hukum...,https://twitter.com/web/statuses/2098092101775...,Negative,@mochmasykur12
4,DOC-00013,RT Begini Dengan Puan atau Ganjar atau Pramono...,begini dengan puan atau ganjar atau pramono an...,begini dengan puan atau ganjar atau pramono a...,https://twitter.com/web/statuses/2098092007784...,Negative,@JapraRahma27808


# SEL 4: Klasifikasi Media

In [27]:
# Daftar platform yang dianggap "Media Sosial"
social_platforms = ['instagram', 'twitter', 'x.com', 'threads', 'youtube', 'tiktok', 'facebook']


def classify_media(media_value):
    """Mengklasifikasikan nilai kolom 'Media' menjadi 'Media Sosial' atau 'Berita/Online'."""
    if not isinstance(media_value, str) or media_value.strip() == "":
        return "Tidak Diketahui"
    media_lower = media_value.strip().lower()
    for platform in social_platforms:
        if platform in media_lower:
            return "Media Sosial"
    return "Berita/Online"


df_clean['Media_Category'] = df_clean['Media'].apply(classify_media)

df_news = df_clean[df_clean['Media_Category'] == 'Berita/Online'].copy()
df_social = df_clean[df_clean['Media_Category'] == 'Media Sosial'].copy()

print(f"Total data Berita/Online : {len(df_news)}")
print(f"Total data Media Sosial  : {len(df_social)}")
print(f"Total data Tidak Diketahui: {(df_clean['Media_Category'] == 'Tidak Diketahui').sum()}")

# Urutkan berdasarkan 'Followers' (proxy reach) sebelum ambil top 5.
sort_col = 'Followers' if 'Followers' in df_clean.columns else None

if sort_col:
    df_news_sorted = df_news.sort_values(by=sort_col, ascending=False, na_position='last')
    df_social_sorted = df_social.sort_values(by=sort_col, ascending=False, na_position='last')
else:
    df_news_sorted = df_news
    df_social_sorted = df_social

top5_news = df_news_sorted.head(5)
top5_social = df_social_sorted.head(5)

selected_df = pd.concat([top5_news, top5_social], ignore_index=True)

# Preview seleksi 5 berita + 5 media sosial yang akan dianalisis
available_preview_cols = [
    col for col in ['No', 'Media_Category', 'Media', 'Headline', 'Mentions', 'Sentiment', 'Author']
    if col in selected_df.columns
]
display(selected_df[available_preview_cols])

Total data Berita/Online : 1146
Total data Media Sosial  : 2720
Total data Tidak Diketahui: 0


,No,Media_Category,Media,Headline,Mentions,Sentiment,Author
0,4,Berita/Online,News,Tiorita: Pemkab Langkat Siap Kolaborasi Dukung...,"FOTO BERSAMA : Pelaksana Tugas Bupati Langkat,...",Positive,bbsnews.id
1,9,Berita/Online,News,Prabowo Perintahkan TNI AL Optimalkan Pencaria...,Sebanyak 11 kapal gabungan telah diterjunkan u...,Neutral,news.republika.co.id
2,18,Berita/Online,News,Puan Maharani Desak Pencarian Maksimal! 5 Jurn...,PUAN MAHARANI: Ketua DPR RI Puan Maharani mend...,Positive,sinfonews.com
3,23,Berita/Online,News,Becak Listrik Bantuan Prabowo Mulai Dijajal di...,Becak listrik bantuan Presiden Prabowo Subiant...,Neutral,jabar.tribunnews.com
4,30,Berita/Online,News,"Amien Rais Singgung Prabowo: Pidatonya Galak, ...",RakyatPos.id &ndash; Ketua Dewan Pertimbangan ...,Neutral,www.rakyatpos.id
5,13,Media Sosial,Youtube,PRABOWO MANTAP&quot;PENYERAHAN BANTUAN BECA LI...,PRABOWO MANTAP&quot;PENYERAHAN BANTUAN BECA LI...,Positive,SAHABAT KANG DEDI MULYADI69
6,14,Media Sosial,Twitter,NaN,@RA_leather ketika dirugikan atas perbuatan me...,Negative,@mochmasykur12
7,15,Media Sosial,Twitter,NaN,RT Begini Dengan Puan atau Ganjar atau Pramono...,Negative,@JapraRahma27808
8,16,Media Sosial,Youtube,Crop shirt linen Wowo ❤️ #music #fashiontrends...,Crop shirt linen Wowo ❤️ #music #fashiontrends...,Positive,Life wth Pv
9,17,Media Sosial,Twitter,NaN,RT momen hanya prabowo yang tidak disalami put...,Negative,@kyryl__


# SEL 5: Output Phase 0

In [21]:
# =========================================================
# SEL 5 (fix): Paksa tepat 5 isu per kategori + perbaiki error
# handling di build_final_table (bug 'Followers')
# =========================================================

import json
import pandas as pd
from IPython.display import display, HTML

def build_structured_text(selected_df):
    lines = []
    for _, row in selected_df.iterrows():
        no = row.get('No', '-')
        kategori = row.get('Media_Category', '-')
        media = row.get('Media', '-')
        headline = row.get('Headline', '-') if pd.notna(row.get('Headline', None)) else '-'
        mentions = row.get('Mentions', '-') if pd.notna(row.get('Mentions', None)) else '-'
        lines.append(
            f"[No: {no} | Kategori: {kategori} | Media: {media}]\n"
            f"Headline: {headline}\n"
            f"Mentions: {mentions}"
        )
    return "\n\n---\n\n".join(lines)

system_prompt = """
You're Syahganda Nainggolan — seorang intelektual-aktivis Indonesia, mantan aktivis ITB era 80-an
yang di-DO karena menentang Orde Baru, meraih gelar Doktor di bidang perburuhan dari FISIP UI,
dan dikenal sebagai tokoh oposisi yang konsisten mengkritik kekuasaan yang sedang berkuasa.
Kamu adalah salah satu petinggi KAMI (Koalisi Aksi Menyelamatkan Indonesia), pendiri
Sabang-Merauke Circle (SMC), dan punya concern besar pada isu buruh, kesejahteraan sosial,
dan nasionalisme.

Cara berpikirmu:
- Kritis dan skeptis terhadap narasi resmi pemerintah/elite penguasa.
- Tajam membedah ketimpangan sosial, nasib buruh/rakyat kecil, dan ancaman terhadap nasionalisme.
- Lugas, berani, dan konfrontatif.
- Mencari pola dan motif di balik data, bukan sekadar mendeskripsikan permukaan teks.

Kamu akan diberikan 5 item dari Berita/Online dan 5 item dari Media Sosial, masing-masing
memiliki identifier 'No'. Analisis data tersebut dan buatlah 2 kelompok analisis isu utama
yang terpisah secara kritis.
"""

def create_prompt(structured_text, n_social, n_news):
    full_prompt = f"""{system_prompt}

=== DATA TERPILIH ({n_news} BERITA/ONLINE + {n_social} MEDIA SOSIAL) ===
{structured_text}
=== AKHIR DATA ===

ATURAN WAJIB (harus dipatuhi, tidak boleh dilanggar):
1. Array "media_sosial" HARUS berisi TEPAT {n_social} objek — TIDAK BOLEH kurang dari {n_social},
   TIDAK BOLEH lebih.
2. Array "berita_online" HARUS berisi TEPAT {n_news} objek — TIDAK BOLEH kurang dari {n_news},
   TIDAK BOLEH lebih.
3. Jika jumlah isu benar-benar berbeda "layak dibahas" kurang dari jumlah wajib, tetap paksakan
   sampai tepat {n_social}/{n_news} dengan memecah sudut pandang/sub-isu dari data yang sama,
   JANGAN mengurangi jumlah array.
4. 'sumber_no' HARUS berisi nilai 'No' yang benar-benar ada di data yang diberikan.
5. 'ringkasan' TIDAK LEBIH dari 3 kalimat.

Jawab HANYA dalam format JSON dengan struktur persis seperti di bawah ini (tanpa teks
pembuka/penutup, tanpa markdown code fence):
{{
  "media_sosial": [
    {{
      "no": 1,
      "isu_utama": "Judul isu media sosial yang tajam",
      "media_pendukung": "Nama platform medsos",
      "ringkasan": "Ringkasan max 3 kalimat...",
      "sumber_no": [nomor 'No' terkait, bisa lebih dari satu]
    }}
  ],
  "berita_online": [
    {{
      "no": 1,
      "isu_utama": "Judul isu berita online yang tajam",
      "media_pendukung": "Nama portal berita",
      "ringkasan": "Ringkasan max 3 kalimat...",
      "sumber_no": [nomor 'No' terkait, bisa lebih dari satu]
    }}
  ]
}}
"""
    return full_prompt


# ---------------------------------------------------------
# Scoring sentimen -> skor 0-1 + label kesimpulan
# ---------------------------------------------------------
KONTROVERSIAL_THRESHOLD = 0.2

def score_sentiment(no_list, source_df):
    if not isinstance(no_list, list) or len(no_list) == 0:
        return None, "Tidak Diketahui"
    if 'Sentiment' not in source_df.columns:
        return None, "Tidak Diketahui"

    subset = source_df[source_df['No'].isin(no_list)]
    if subset.empty:
        return None, "Tidak Diketahui"

    s = subset['Sentiment'].astype(str).str.strip().str.lower()
    pos = s.isin(['positif', 'positive', 'pos']).sum()
    neg = s.isin(['negatif', 'negative', 'neg']).sum()
    total = pos + neg

    if total == 0:
        return None, "Tidak Diketahui"

    score = round(pos / total, 2)
    diff_ratio = abs(pos - neg) / total

    if diff_ratio <= KONTROVERSIAL_THRESHOLD:
        label = "Kontroversial"
    elif pos > neg:
        label = "Positif"
    else:
        label = "Negatif"

    return score, label


# ---------------------------------------------------------
# Key Opinion Leader -> Author non-news (Media Sosial) dengan
# Followers tertinggi. DIPERBAIKI: aman walau kolom 'Followers'
# tidak ada / berisi NaN / campur tipe data.
# ---------------------------------------------------------
def find_key_opinion_leader(no_list, source_df):
    if not isinstance(no_list, list) or len(no_list) == 0:
        return "Tidak Diketahui"

    required_cols = {'No', 'Media_Category', 'Author'}
    if not required_cols.issubset(source_df.columns):
        return "Tidak Diketahui"

    subset = source_df[source_df['No'].isin(no_list)]
    subset_social = subset[subset['Media_Category'] == 'Media Sosial'].copy()

    if subset_social.empty:
        return "Tidak ada (isu didominasi sumber berita)"

    if 'Followers' in subset_social.columns:
        # Paksa numerik: nilai non-numerik/NaN jadi -1 supaya tidak error saat sort,
        # dan otomatis kalah urutan dibanding follower yang valid.
        subset_social['_followers_num'] = pd.to_numeric(
            subset_social['Followers'], errors='coerce'
        ).fillna(-1)
        subset_social_sorted = subset_social.sort_values(by='_followers_num', ascending=False)
        top_row = subset_social_sorted.iloc[0]
        author = top_row.get('Author', 'Tidak Diketahui')
        followers_val = top_row.get('Followers', None)

        if pd.notna(followers_val):
            try:
                return f"{author} ({int(float(followers_val)):,} followers)".replace(",", ".")
            except (ValueError, TypeError):
                return str(author)
        return f"{author} (followers tidak diketahui)"
    else:
        # Kolom 'Followers' memang tidak ada di source_df -> ambil author pertama saja
        author = subset_social.iloc[0].get('Author', 'Tidak Diketahui')
        return f"{author} (data followers tidak tersedia)"


# ---------------------------------------------------------
# Politically Exposed Person -> panggilan Gemini kedua per isu
# ---------------------------------------------------------
def extract_pep(summary_text):
    if not isinstance(summary_text, str) or summary_text.strip() == "" or summary_text == "-":
        return "Tidak ada tokoh spesifik"

    pep_prompt = f"""Dari ringkasan berikut, identifikasi HANYA nama tokoh publik/politik
(politically exposed person) yang paling menjadi sorotan atau paling banyak dibahas.

Ringkasan:
\"\"\"{summary_text}\"\"\"

Instruksi:
- Jawab HANYA dengan nama tokoh (boleh lebih dari satu, pisahkan dengan koma).
- Jangan beri penjelasan tambahan, tanda kutip, atau kalimat pembuka/penutup.
- Jika tidak ada tokoh publik/politik spesifik yang disebut, jawab persis: Tidak ada tokoh spesifik
"""
    result = get_gemini_response(pep_prompt)
    return result.strip() if result else "Tidak ada tokoh spesifik"


# ---------------------------------------------------------
# Rakit tabel final -> DIPERBAIKI: per-baris dibungkus try/except
# sendiri, supaya 1 baris error tidak menggagalkan seluruh tabel.
# ---------------------------------------------------------
def build_final_table(df_issues, source_df):
    cols = ["no", "nama_isu", "sentimen", "key_opinion_leader", "summary", "politically_exposed_person"]
    if df_issues.empty:
        return pd.DataFrame(columns=cols)

    rows = []
    for idx, row in df_issues.iterrows():
        try:
            no_list = row.get("sumber_no", [])
            ringkasan = row.get("ringkasan", "-")

            score, label = score_sentiment(no_list, source_df)
            sentimen_display = f"{score} ({label})" if score is not None else label

            kol = find_key_opinion_leader(no_list, source_df)
            pep = extract_pep(ringkasan)

            rows.append({
                "no": row.get("no", idx + 1),
                "nama_isu": row.get("isu_utama", "-"),
                "sentimen": sentimen_display,
                "key_opinion_leader": kol,
                "summary": ringkasan,
                "politically_exposed_person": pep,
            })
        except Exception as e:
            print(f"⚠️ Gagal memproses baris ke-{idx} ('{row.get('isu_utama', '?')}'): {e}")
            rows.append({
                "no": row.get("no", idx + 1),
                "nama_isu": row.get("isu_utama", "-"),
                "sentimen": "Error",
                "key_opinion_leader": "Error",
                "summary": row.get("ringkasan", "-"),
                "politically_exposed_person": "Error",
            })

    return pd.DataFrame(rows)


# ---------------------------------------------------------
# Bangun prompt & panggil Gemini, dengan validasi jumlah isu
# (auto-retry sekali kalau jumlahnya belum 5 & 5)
# ---------------------------------------------------------
N_SOCIAL = 5
N_NEWS = 5

structured_text = build_structured_text(selected_df)
final_prompt = create_prompt(structured_text, N_SOCIAL, N_NEWS)

def call_and_parse(prompt):
    raw = get_gemini_response(prompt)
    clean = raw.strip()
    if clean.startswith("```"):
        clean = clean.split("```")[1]
        if clean.lower().startswith("json"):
            clean = clean[4:]
        clean = clean.strip()
    return json.loads(clean), raw

result_json, raw_response = None, ""
for attempt in range(1, 3):  # coba max 2x kalau jumlah isu belum sesuai
    try:
        result_json, raw_response = call_and_parse(final_prompt)
        n_social_got = len(result_json.get("media_sosial", []))
        n_news_got = len(result_json.get("berita_online", []))

        if n_social_got == N_SOCIAL and n_news_got == N_NEWS:
            break
        else:
            print(f"⚠️ Percobaan {attempt}: dapat {n_social_got} isu medsos & {n_news_got} isu berita "
                  f"(target {N_SOCIAL} & {N_NEWS}). Mencoba lagi...")
    except json.JSONDecodeError as e:
        print(f"⚠️ Percobaan {attempt}: gagal parsing JSON ({e}). Mencoba lagi...")

if result_json is None:
    print("⚠️ Gagal mendapatkan hasil yang valid dari Gemini setelah beberapa percobaan.")
    print("Raw response terakhir:")
    print(raw_response)
else:
    print("\n=== TABEL 1: ANALISIS ISU MEDIA SOSIAL ===")
    df_medsos_raw = pd.DataFrame(result_json.get("media_sosial", []))
    df_medsos = build_final_table(df_medsos_raw, selected_df)
    display(df_medsos)

    print("\n=== TABEL 2: ANALISIS ISU BERITA ONLINE ===")
    df_berita_raw = pd.DataFrame(result_json.get("berita_online", []))
    df_berita = build_final_table(df_berita_raw, selected_df)
    display(df_berita)

⚠️ Model 'gemini-flash-latest' percobaan 1/3 gagal (503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently exp...). Coba lagi dalam 6.3 detik...
⚠️ Model 'gemini-flash-latest' percobaan 2/3 gagal (503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently exp...). Coba lagi dalam 10.8 detik...
⚠️ Model 'gemini-flash-latest' gagal setelah 3 percobaan (503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently exp...). Lanjut ke model berikutnya...
⚠️ Model 'gemini-2.5-flash' tidak tersedia (404). Lanjut ke model berikutnya...

=== TABEL 1: ANALISIS ISU MEDIA SOSIAL ===
⚠️ Model 'gemini-flash-latest' percobaan 1/3 gagal (503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently exp...). Coba lagi dalam 7.0 detik...
⚠️ Model 'gemini-flash-latest' percobaan 2/3 gagal (503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently exp...). Coba lagi dalam 11.5 detik...
⚠️ Model 'gemini-flash-latest' g

,no,nama_isu,sentimen,key_opinion_leader,summary,politically_exposed_person
0,1,Politik Citra dan Instrumentalisasi Bantuan So...,1.0 (Positif),SAHABAT KANG DEDI MULYADI69 (data followers ti...,Pemberian bantuan becak listrik di Sumedang di...,Tidak ada tokoh spesifik
1,2,Degradasi Nalar Kritis dan Budaya 'Amplop',0.0 (Negatif),@mochmasykur12 (data followers tidak tersedia),Munculnya narasi yang merendahkan pihak yang m...,Tidak ada tokoh spesifik
2,3,Politik Transaksional dan Perebutan Pengaruh Elit,0.0 (Negatif),@JapraRahma27808 (data followers tidak tersedia),Wacana koalisi antar-partai hanya dilihat dari...,Tidak ada tokoh spesifik
3,4,Pendangkalan Isu Politik Menjadi Tren Fashion,1.0 (Positif),Life wth Pv (data followers tidak tersedia),Politik nasional kini didangkalkan menjadi kon...,Tidak ada tokoh spesifik
4,5,Kemerosotan Wibawa Diplomatik Indonesia di Mat...,0.0 (Negatif),@kyryl__ (data followers tidak tersedia),Viralnya momen diplomasi yang dianggap blunder...,Tidak ada tokoh spesifik



=== TABEL 2: ANALISIS ISU BERITA ONLINE ===
⚠️ Model 'gemini-flash-latest' percobaan 1/3 gagal (503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently exp...). Coba lagi dalam 6.9 detik...
⚠️ Model 'gemini-flash-latest' percobaan 2/3 gagal (503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently exp...). Coba lagi dalam 11.4 detik...
⚠️ Model 'gemini-flash-latest' percobaan 1/3 gagal (429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your cu...). Coba lagi dalam 5.3 detik...
⚠️ Model 'gemini-flash-latest' percobaan 2/3 gagal (429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your cu...). Coba lagi dalam 11.2 detik...
⚠️ Model 'gemini-flash-latest' gagal setelah 3 percobaan (429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your cu...). Lanjut ke model berikutnya...
⚠️ Model 'gemini-2.5-flash' tidak tersedia (404). Lanjut ke model berikutnya...
⚠️ Model 'gemini-flash-latest' 

,no,nama_isu,sentimen,key_opinion_leader,summary,politically_exposed_person
0,1,Pencitraan 'Ketahanan Pangan' di Balik Penjara,1.0 (Positif),Tidak ada (isu didominasi sumber berita),Program pemberdayaan warga binaan di Langkat r...,Tidak ada tokoh spesifik
1,2,Kegagalan Sistem Penyelamatan dan Perlindungan...,Tidak Diketahui,Tidak ada (isu didominasi sumber berita),Hilangnya jurnalis di Selat Sunda menyingkap l...,Tidak ada tokoh spesifik
2,3,Hipokrisi Elit dalam Merespons Tragedi Kemanus...,1.0 (Positif),Tidak ada (isu didominasi sumber berita),Desakan petinggi legislatif agar pencarian jur...,Tidak ada tokoh spesifik
3,4,Manipulasi Populis Melalui Bantuan Teknis Murahan,Tidak Diketahui,Tidak ada (isu didominasi sumber berita),Pembagian becak listrik kepada pengemudi lansi...,Tidak ada tokoh spesifik
4,5,Sandiwara 'Jocowisasi' dalam Rezim Prabowo,Tidak Diketahui,Tidak ada (isu didominasi sumber berita),Kritik tajam mengenai Prabowo yang masih terje...,"Prabowo, Jokowi"
